# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a worked example for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library, which leverages the Croissant schema.

### Dataset Source
The dataset source metadata is accessed via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a DatasetMetadata object
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")


## 2. Data Overview

Review all available record sets, fields, and their `@id` values. These unique identifiers are essential for referencing and extracting data.


In [ ]:
# List record sets and their corresponding fields/columns by @id
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print('Record Sets:')
    for rs in metadata.record_sets:
        print(f"- RecordSet @id: {rs.id if hasattr(rs,'id') else getattr(rs, '@id', None)}   name: {getattr(rs,'name', None)}")
        print('  Fields:')
        # The fields are a list of FieldMetadata
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    - Field @id: {field.id if hasattr(field,'id') else getattr(field, '@id', None)}, name: {getattr(field,'name', None)}, data_type: {getattr(field, 'data_type', None)}")
        print('  Columns:')
        if hasattr(rs, 'columns'):
            for col in rs.columns:
                print(f"    - Column @id: {col.id if hasattr(col,'id') else getattr(col, '@id', None)}, name: {getattr(col,'name', None)}, data_type: {getattr(col, 'data_type', None)}")
else:
    print("No record sets found in this dataset's metadata.")

## 3. Data Extraction

Load data from each record set into Pandas DataFrames for further analysis. Use the record set and field/column `@id`s from the previous section for referencing precisely.

In [ ]:
# Identify all record set @id values from the metadata
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None) for rs in metadata.record_sets]
    print('Available Record Set @ids:', record_set_ids)
else:
    record_set_ids = []

# Load each available record set (by @id) into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records for RecordSet @id: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} records.')
        print('Columns:', dataframes[record_set_id].columns.tolist())
    else:
        print('No records found for this record set.')

# For illustration, select the first available record set if present
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f'Example head of DataFrame for RecordSet @id: {main_record_set_id}')
    display(dataframes[main_record_set_id].head())
else:
    print('No tabular dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic processing and statistical exploration on a numeric field. We will:
* Select a field to filter and normalize
* Remove outliers (if any),
* Normalize numeric variables,
* Group by a categorical field (where available)

Remember to reference columns by their `@id`.


In [ ]:
# Replace these with actual field @ids from Section 2 above
if dataframes:
    df = dataframes[main_record_set_id]
    print('First few columns:', df.columns.tolist())

    # For illustration: Try to select a numeric field @id
    # Example: use any column name that has 'age' in it (update as appropriate)
    numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower()]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
    else:
        numeric_field_id = df.columns[0]  # fallback

    print(f'Using numeric field for filtering and normalization: "{numeric_field_id}"')

    # Choose a threshold based on the field (arbitrary example)
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
    filt = pd.Series([True]*len(df))
    if threshold is not None:
        filt = df[numeric_field_id] > threshold
    filtered_df = df[filt].copy()
    print(f"Filtered records with '{numeric_field_id}' > {threshold if threshold is not None else '(N/A)'}:")
    display(filtered_df.head())

    # Normalize the numeric field
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col]))]

    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f'Grouping by field: {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f'{numeric_field_id}_mean')
        display(grouped_df.head())
    else:
        print('No suitable categorical field found for grouping.')
else:
    print('No data frame available for EDA.')

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship to the grouping variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Histogram of the numeric variable
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by grouping field (if available)
    if group_field_candidates:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to load, examine, and process the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset using the Croissant schema and `mlcroissant`. Using `@id` references for data structures, we efficiently explored structure, performed basic filtering and normalization, and produced summary plots. This approach enables reproducible and robust exploration of FAIR datasets.
